# Livro para consulta:
- https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html
- https://jakevdp.github.io/PythonDataScienceHandbook/03.09-pivot-tables.html
    

# 1. Importando bibliotecas <a name="import"></a>

<div style="text-align: right"
     
[Voltar ao índice](#Contents)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

# 2. Carregando o dataframe SINASC <a name="read"></a>
<div style="text-align: right"
     
[Voltar ao índice](#Contents)

In [8]:
df = pd.read_csv('SINASC_RO_2019.csv')
df.shape

(27028, 69)

# Tarefa 1

### 1. Idade media das mães e dos pais por município (coluna munResNome)


In [13]:
print(df[['IDADEMAE', 'IDADEPAI', 'munResNome']].head())

   IDADEMAE  IDADEPAI               munResNome
0        19      26.0    Alta Floresta D'Oeste
1        29      24.0    Alta Floresta D'Oeste
2        37      32.0    Alta Floresta D'Oeste
3        30      24.0  Alto Alegre dos Parecis
4        30      27.0    Alta Floresta D'Oeste


In [11]:
print("Valores nulos por coluna:")
print(df[['IDADEMAE', 'IDADEPAI']].isnull().sum())

Valores nulos por coluna:
IDADEMAE        0
IDADEPAI    19421
dtype: int64


In [12]:
df['munResNome'].nunique()

53

In [9]:
porcentagem_pais_faltantes = round((df['IDADEPAI'].isnull().sum() / len(df)) * 100, 2)
print(f"Percentual de dados faltantes em IDADEPAI: {porcentagem_pais_faltantes}%")


Percentual de dados faltantes em IDADEPAI: 71.86%


In [10]:
idades_medias = df.groupby('munResNome')[['IDADEMAE', 'IDADEPAI']].mean().round(1)
print(idades_medias)

                           IDADEMAE  IDADEPAI
munResNome                                   
Alta Floresta D'Oeste          26.0      29.5
Alto Alegre dos Parecis        24.8      29.2
Alto Paraíso                   25.0      28.8
Alvorada D'Oeste               25.8      30.8
Ariquemes                      25.6      32.5
Buritis                        25.6      30.8
Cabixi                         26.0      34.3
Cacaulândia                    25.5      36.3
Cacoal                         26.9      30.9
Campo Novo de Rondônia         24.8      30.5
Candeias do Jamari             25.2      29.9
Castanheiras                   27.3      30.4
Cerejeiras                     27.2      31.7
Chupinguaia                    25.3      29.6
Colorado do Oeste              27.6      34.4
Corumbiara                     24.7      32.5
Costa Marques                  24.4      30.6
Cujubim                        24.5      31.4
Espigão D'Oeste                26.1      31.2
Governador Jorge Teixeira      24.

### 2. Peso médio dos bebes por sexo que nasceram no dia do seu aniversário por faixas de escolaridade mae
Ex: Você, aluna(o), nasceu no dia 10/01, então você precisa filtrar o conjunto de dados nessa data e calcular o peso médio dos bebês de cada sexo por faixa de escolaridade da mãe.

In [16]:
df_aniversario = df[df['DTNASC'] == '2019-02-22']
print(f"Total de registros em 22/02: {len(df_aniversario)}")

Total de registros em 22/02: 85


In [17]:
print("Média de PESO por SEXO separado por ESCOLARIDADE:")

resultado = df_aniversario.groupby(['ESCMAE', 'SEXO'])['PESO'].mean().round(2)
print(resultado)

Média de PESO por SEXO separado por ESCOLARIDADE:
ESCMAE           SEXO     
1 a 3 anos       Masculino    3350.00
12 anos ou mais  Feminino     3439.71
                 Masculino    3116.12
4 a 7 anos       Feminino     3424.17
                 Masculino    3196.25
8 a 11 anos      Feminino     3254.44
                 Masculino    3190.67
Name: PESO, dtype: float64


In [18]:
print("Tabela Dinâmica (pivot):")

tabela_media = pd.pivot_table(df_aniversario, values='PESO', index='ESCMAE', columns='SEXO', aggfunc='mean', fill_value=0).round(2)

tabela_contagem = pd.pivot_table(df_aniversario, values='PESO', index='ESCMAE', columns='SEXO', aggfunc='count', fill_value=0)

print("MÉDIA DE PESO (gramas) por ESCOLARIDADE e SEXO:")
print(tabela_media)
print("\nQUANTIDADE DE REGISTROS por ESCOLARIDADE e SEXO:")
print(tabela_contagem)

Tabela Dinâmica (pivot):
MÉDIA DE PESO (gramas) por ESCOLARIDADE e SEXO:
SEXO             Feminino  Masculino
ESCMAE                              
1 a 3 anos           0.00    3350.00
12 anos ou mais   3439.71    3116.12
4 a 7 anos        3424.17    3196.25
8 a 11 anos       3254.44    3190.67

QUANTIDADE DE REGISTROS por ESCOLARIDADE e SEXO:
SEXO             Feminino  Masculino
ESCMAE                              
1 a 3 anos              0          3
12 anos ou mais         7          8
4 a 7 anos              6          4
8 a 11 anos            27         30


### 3. Qual o municipio que nasceu menos bebe em 2019?
    - qual a idade media, maxima, minima das maes nesse municipio?
    - qual a idade media, maxima, minima dos pais nesse municipio?

In [19]:
print("Quantidade de nascimentos por município:")
contagem_municipios = df['munResNome'].value_counts()

print("\nMunicipio com menos nascimentos:")
print(contagem_municipios.tail(10))


Quantidade de nascimentos por município:

Municipio com menos nascimentos:
munResNome
Cabixi                     80
Cacaulândia                75
Teixeirópolis              64
São Felipe D'Oeste         54
Rio Crespo                 50
Parecis                    44
Primavera de Rondônia      43
Pimenteiras do Oeste       40
Castanheiras               32
Município ignorado - RO     1
Name: count, dtype: int64


In [20]:
dado_faltante = contagem_municipios[contagem_municipios == 1].index[0]
df_3 = df[df['munResNome'] != dado_faltante]

contagem_municipios = df_3['munResNome'].value_counts()
menos_nascimento = contagem_municipios.idxmin()
print(f"\nMunicipio com menos nascimentos: {menos_nascimento}")
print(f"Quantidade de nascimentos: {contagem_municipios[menos_nascimento]}")


Municipio com menos nascimentos: Castanheiras
Quantidade de nascimentos: 32


In [21]:
df_3 = df_3[df_3['munResNome'] == menos_nascimento]

print ("\nESTATÍSTICAS DE IDADE:")
print("_" * 23)

#Idade da mãe
idades_mae = df_3['IDADEMAE'].replace(0, np.nan).dropna()
print(f"\nIdade da Mãe:")
print(f" Média: {idades_mae.mean():.1f} anos")
print(f" Máxima: {idades_mae.max():.0f} anos")
print(f" Mínima: {idades_mae.min():.0f} anos")


#Idade do pai
idades_pai = df_3['IDADEPAI'].replace(0, np.nan).dropna()
print(f"\nIdade do Pai:")
print(f" Média: {idades_pai.mean():.1f} anos")
print(f" Máxima: {idades_pai.max():.0f} anos")
print(f" Mínima: {idades_pai.min():.0f} anos")





ESTATÍSTICAS DE IDADE:
_______________________

Idade da Mãe:
 Média: 27.3 anos
 Máxima: 39 anos
 Mínima: 17 anos

Idade do Pai:
 Média: 30.4 anos
 Máxima: 43 anos
 Mínima: 17 anos


### 4. Qual o municipio que nasceu mais bebe no mês de março?
    - qual a quantidade de filhos vivos media, maxima, minima nesse municipio?
    - qual a idade media, maxima, minima dos pais nesse municipio?



In [22]:
df_marco = df[df['DTNASC'].str.startswith('2019-03')]
print(f"Total de nascimentos em MARÇO: {len(df_marco)}")
print(f"Perentual de nascimentos em MARÇO: {(len(df_marco)/len(df)*100):.1f}%")

Total de nascimentos em MARÇO: 2456
Perentual de nascimentos em MARÇO: 9.1%


In [23]:
contagem_marco = df_marco['munResNome'].value_counts()
print(f"Média de nascimentos por município em MARÇO: {contagem_marco.mean():.1f}")
print(f"Municipio com mais nascimentos em MARÇO: {contagem_marco.idxmax()}")
print(f"Quantidade de nascimentos de MARÇO em {contagem_marco.idxmax()}: {contagem_marco.max()}")

Média de nascimentos por município em MARÇO: 47.2
Municipio com mais nascimentos em MARÇO: Porto Velho
Quantidade de nascimentos de MARÇO em Porto Velho: 744


In [29]:
df_4 = df[df['munResNome'] == contagem_marco.idxmax()]
df_4.shape

(8437, 69)

In [33]:
df_4.columns


Index(['ORIGEM', 'CODESTAB', 'CODMUNNASC', 'LOCNASC', 'IDADEMAE', 'ESTCIVMAE',
       'ESCMAE', 'CODOCUPMAE', 'QTDFILVIVO', 'QTDFILMORT', 'CODMUNRES',
       'GESTACAO', 'GRAVIDEZ', 'PARTO', 'CONSULTAS', 'DTNASC', 'HORANASC',
       'SEXO', 'APGAR1', 'APGAR5', 'RACACOR', 'PESO', 'IDANOMAL', 'DTCADASTRO',
       'CODANOMAL', 'NUMEROLOTE', 'VERSAOSIST', 'DTRECEBIM', 'DIFDATA',
       'DTRECORIGA', 'NATURALMAE', 'CODMUNNATU', 'CODUFNATU', 'ESCMAE2010',
       'SERIESCMAE', 'DTNASCMAE', 'RACACORMAE', 'QTDGESTANT', 'QTDPARTNOR',
       'QTDPARTCES', 'IDADEPAI', 'DTULTMENST', 'SEMAGESTAC', 'TPMETESTIM',
       'CONSPRENAT', 'MESPRENAT', 'TPAPRESENT', 'STTRABPART', 'STCESPARTO',
       'TPNASCASSI', 'TPFUNCRESP', 'TPDOCRESP', 'DTDECLARAC', 'ESCMAEAGR1',
       'STDNEPIDEM', 'STDNNOVA', 'CODPAISRES', 'TPROBSON', 'PARIDADE',
       'KOTELCHUCK', 'CONTADOR', 'munResStatus', 'munResTipo', 'munResNome',
       'munResUf', 'munResLat', 'munResLon', 'munResAlt', 'munResArea'],
      dtype='object')

In [34]:
df_4['QTDFILVIVO'].isnull().sum()

np.int64(1118)

In [35]:
filhos_vivos = df_4['QTDFILVIVO'].replace(0, np.nan).dropna()
print(f"\nQuantidade de filhos vivos:")
print(f" Média: {filhos_vivos.mean():.1f}")
print(f" Máxima: {filhos_vivos.max():.0f}")
print(f" Mínima: {filhos_vivos.min():.0f}")



Quantidade de filhos vivos:
 Média: 1.7
 Máxima: 12
 Mínima: 1


In [30]:
print (f"\nESTATÍSTICAS DE IDADE de {contagem_marco.idxmax()}:")
print("_" * 40)

#Idade da Mãe
if 'IDADEMAE' in df_4.columns:
    idade_mae = df_4['IDADEMAE'].replace(0, np.nan).dropna()
    print(f"\nIdade da Mãe:")
    print(f" Média: {idade_mae.mean():.1f} anos")
    print(f" Máxima: {idade_mae.max():.0f} anos")
    print(f" Mínima: {idade_mae.min():.0f} anos")
    print(f" Total de registros válidos: {len(idade_mae)}")

#Idade do Pai
if 'IDADEPAI' in df_4.columns:
    idade_pai = df_4['IDADEPAI'].replace(0, np.nan).dropna()
    print(f"\nIdade do Pai:")
    print(f" Média: {idade_pai.mean():.1f} anos")
    print(f" Máxima: {idade_pai.max():.1f} anos")
    print(f" Mínima: {idade_pai.min():.1f} anos")
    print(f" Total de registros válidos: {len(idade_pai)}")






ESTATÍSTICAS DE IDADE de Porto Velho:
________________________________________

Idade da Mãe:
 Média: 26.3 anos
 Máxima: 47 anos
 Mínima: 12 anos
 Total de registros válidos: 8437

Idade do Pai:
 Média: 32.4 anos
 Máxima: 65.0 anos
 Mínima: 16.0 anos
 Total de registros válidos: 672


In [31]:
pais_faltantes_4 = round((df_4['IDADEPAI'].isnull().sum() / len(df_4)) * 100, 2)
print(f"Percentual de dados faltantes em IDADEPAI: {pais_faltantes_4}%")

Percentual de dados faltantes em IDADEPAI: 92.04%


In [28]:
print (f"\nESTATÍSTICAS DE IDADE RORAIMA:")
print("_" * 40)

#Idade da Mãe
if 'IDADEMAE' in df.columns:
    idade_mae = df['IDADEMAE'].replace(0, np.nan).dropna()
    print(f"\nIdade da Mãe:")
    print(f" Média: {idade_mae.mean():.1f} anos")
    print(f" Máxima: {idade_mae.max():.0f} anos")
    print(f" Mínima: {idade_mae.min():.0f} anos")
    print(f" Total de registros válidos: {len(idade_mae)}")

#Idade do Pai
if 'IDADEPAI' in df.columns:
    idade_pai = df['IDADEPAI'].replace(0, np.nan).dropna()
    print(f"\nIdade do Pai:")
    print(f" Média: {idade_pai.mean():.1f} anos")
    print(f" Máxima: {idade_pai.max():.1f} anos")
    print(f" Mínima: {idade_pai.min():.1f} anos")
    print(f" Total de registros válidos: {len(idade_pai)}")






ESTATÍSTICAS DE IDADE RORAIMA:
________________________________________

Idade da Mãe:
 Média: 26.1 anos
 Máxima: 53 anos
 Mínima: 11 anos
 Total de registros válidos: 27028

Idade do Pai:
 Média: 31.1 anos
 Máxima: 86.0 anos
 Mínima: 15.0 anos
 Total de registros válidos: 7607


In [32]:
print ("ESTATÍSTICAS DESCRITIVAS COMPLETAS:")
print(df_4[['IDADEMAE', 'IDADEPAI']].describe().round(1))

ESTATÍSTICAS DESCRITIVAS COMPLETAS:
       IDADEMAE  IDADEPAI
count    8437.0     672.0
mean       26.3      32.4
std         6.5       7.9
min        12.0      16.0
25%        21.0      27.0
50%        26.0      32.0
75%        31.0      37.0
max        47.0      65.0


### Analise as respostas encontradas, tire algum insight delas, conte pra gente algo encontrado nos dados. Algo que você julgue relevante e novo pra você.

In [39]:
df_menos_16 = df[df['IDADEMAE'] < 16]
print(f"Total de nascimentos com mãe menor que 16 anos: {len(df_menos_16)}")
print(f"Percentual de nascimentos com mãe menor que 16 anos: {(len(df_menos_16)/len(df)*100):.1f}%")

Total de nascimentos com mãe menor que 16 anos: 552
Percentual de nascimentos com mãe menor que 16 anos: 2.0%


In [44]:
print("\nDistribuição por idade:")
for idade in range(1, 16):
    qtd = len(df_menos_16[df_menos_16['IDADEMAE'] == idade])
    if qtd > 0:
        print(f"Idade {idade}: {qtd} nascimentos")


Distribuição por idade:
Idade 11: 1 nascimentos
Idade 12: 7 nascimentos
Idade 13: 27 nascimentos
Idade 14: 163 nascimentos
Idade 15: 354 nascimentos


In [49]:
df_menos_16['QTDFILVIVO'].value_counts()

,count
QTDFILVIVO,
0.0,447
1.0,30
2.0,1


Exemplo:
- Ah, descobri que a idade mediana das mulheres que deram a luz no ano de 2019 dos municipios x é maior que y.

# INSIGHTS

 - O número de dados faltantes de pais é alarmante. Mais de 70%

 - No município de Porto Velho, onde tiveram mais nascimentos no mês de Março, a média é maior ainda. Chega a 92%

 - Em 2019, 552 crianças menores de 16 anos foram mães em Roraima.
 - Dessas 552 crianças, pelo menos 30 já possuem 1 filho vivo.
